# Stage 5 — Base ASAG Dependency Triples & Relational Structures

**Project:** Automatic Short Answer Grading using Semantic and Dependency Analysis  
**Repository:** `proposition-level-asag`  
**Current Stage:** Stage 5 — Dependency Triples & Relational Structures

---

### Purpose
This notebook demonstrates the **Stage 5 Component** for the Base ASAG system.

Stage 5 bridges the syntactic tree produced in Stage 4 into structured, reusable dependency relations for subsequent matching:
1. **Raw Dependency Arcs:** `(governor, dependency_relation, dependent)` retaining 100% of syntactic context, modifiers, prepositions, and punctuation.
2. **ASAG-Oriented Canonical Relations:** `(subject, predicate, object)` grounded in explicit syntactic paths (e.g. `nsubj` + `dobj`).

> **RESEARCH INTEGRITY & BOUNDARIES:**  
> - Stage 5 does **not** perform semantic similarity or synonym matching. Terms such as `record` and `row` remain distinct lemmas.  
> - Prepositional phrases (e.g., `in a table`) are **not** collapsed into false direct objects.  
> - No similarity scoring or grading is performed.

> **NOTICE: DEMO DATA — NOT RESEARCH DATA**  
> Synthetic demonstration examples for pipeline verification only.

## 1. Imports and Setup

We import pandas and the public pipeline modules:
- Stage 3: `TextPreprocessor`
- Stage 4: `DependencyParser`
- Stage 5: `DependencyTripleExtractor`

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Ensure repository root is on sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from base_asag.preprocessing import TextPreprocessor
from base_asag.dependency import DependencyParser, DependencyTripleExtractor

preprocessor = TextPreprocessor(model_name="en_core_web_sm")
parser = DependencyParser(model_name="en_core_web_sm")
extractor = DependencyTripleExtractor()

print("Initialized Stage 3 Preprocessor, Stage 4 Dependency Parser, and Stage 5 Triple Extractor.")

Initialized Stage 3 Preprocessor, Stage 4 Dependency Parser, and Stage 5 Triple Extractor.


## 2. Parse & Extract Triples for Reference and Student Answers

We process the global demonstration example:
- **Reference Answer:** *"A primary key uniquely identifies each record in a table."*
- **Student Answer:** *"A primary key uniquely identifies every row in a table."*

In [2]:
reference_text = "A primary key uniquely identifies each record in a table."
student_text = "A primary key uniquely identifies every row in a table."

# Full pipeline chaining: Stage 3 -> Stage 4 -> Stage 5
ref_parsed = parser.parse_sentence(reference_text)
stu_parsed = parser.parse_sentence(student_text)

ref_triples = extractor.extract(ref_parsed)
stu_triples = extractor.extract(stu_parsed)

print(f"Reference Arcs Count: {ref_triples.arc_count}, Canonical Relations: {ref_triples.relation_count}")
print(f"Student Arcs Count:   {stu_triples.arc_count}, Canonical Relations: {stu_triples.relation_count}")

Reference Arcs Count: 10, Canonical Relations: 1
Student Arcs Count:   10, Canonical Relations: 1


## 3. Raw Dependency Arcs Table

Every non-root dependency edge in the sentence is captured as a directed arc `(governor, relation, dependent)` preserving both surface strings and base lemmas.

In [3]:
def arcs_to_dataframe(arcs):
    rows = []
    for arc in arcs:
        rows.append({
            "Governor": arc.governor_text,
            "Gov Lemma": arc.governor_lemma,
            "Relation": arc.relation,
            "Dependent": arc.dependent_text,
            "Dep Lemma": arc.dependent_lemma,
            "Dep POS": arc.dependent_pos,
        })
    return pd.DataFrame(rows)

print("--- Reference Answer Raw Dependency Arcs ---")
df_ref_arcs = arcs_to_dataframe(ref_triples.raw_arcs)
df_ref_arcs

--- Reference Answer Raw Dependency Arcs ---


,Governor,Gov Lemma,Relation,Dependent,Dep Lemma,Dep POS
0,key,key,det,A,a,DET
1,key,key,amod,primary,primary,ADJ
2,identifies,identify,nsubj,key,key,NOUN
3,identifies,identify,advmod,uniquely,uniquely,ADV
4,record,record,det,each,each,DET
5,identifies,identify,dobj,record,record,NOUN
6,identifies,identify,prep,in,in,ADP
7,table,table,det,a,a,DET
8,in,in,pobj,table,table,NOUN
9,identifies,identify,punct,.,.,PUNCT


## 4. Student Answer Raw Dependency Arcs Table

In [4]:
print("--- Student Answer Raw Dependency Arcs ---")
df_stu_arcs = arcs_to_dataframe(stu_triples.raw_arcs)
df_stu_arcs

--- Student Answer Raw Dependency Arcs ---


,Governor,Gov Lemma,Relation,Dependent,Dep Lemma,Dep POS
0,key,key,det,A,a,DET
1,key,key,amod,primary,primary,ADJ
2,identifies,identify,nsubj,key,key,NOUN
3,identifies,identify,advmod,uniquely,uniquely,ADV
4,row,row,det,every,every,DET
5,identifies,identify,dobj,row,row,NOUN
6,identifies,identify,prep,in,in,ADP
7,table,table,det,a,a,DET
8,in,in,pobj,table,table,NOUN
9,identifies,identify,punct,.,.,PUNCT


## 5. ASAG-Oriented Canonical Relations (SPO Triples)

Canonical relations extract verified clause-level subject-predicate-object tuples:
- `(subject, predicate, object)`
- `(subject_lemma, predicate_lemma, object_lemma)`

Notice that modifiers like *"uniquely"* (`advmod`) and prepositional attachments (*"in a table"*) remain in raw arcs and are not falsely promoted to objects.

In [5]:
def relations_to_dataframe(relations):
    rows = []
    for rel in relations:
        rows.append({
            "Subject": rel.subject,
            "Predicate": rel.predicate,
            "Object": rel.object,
            "Subject Lemma": rel.subject_lemma,
            "Predicate Lemma": rel.predicate_lemma,
            "Object Lemma": rel.object_lemma,
            "Relation Type": rel.relation_type,
            "Source Arcs": ", ".join(rel.source_dependency_relations),
        })
    return pd.DataFrame(rows)

print("--- Canonical Clause Relations ---")
df_relations = pd.concat([
    relations_to_dataframe(ref_triples.canonical_relations).assign(Source="Reference"),
    relations_to_dataframe(stu_triples.canonical_relations).assign(Source="Student"),
], ignore_index=True)
df_relations

--- Canonical Clause Relations ---


,Subject,Predicate,Object,Subject Lemma,Predicate Lemma,Object Lemma,Relation Type,Source Arcs,Source
0,key,identifies,record,key,identify,record,SPO,"nsubj, dobj",Reference
1,key,identifies,row,key,identify,row,SPO,"nsubj, dobj",Student


## 6. Crucial Stage 5 Research Observation: No Synonym Matching Yet

Look at the extracted lemma tuples:
- **Reference Triple:** `("key", "identify", "record")`
- **Student Triple:**   `("key", "identify", "row")`

At **Stage 5**, `record` and `row` remain strictly distinct. Stage 5 does **not** equate synonyms or compute similarity.
In **Stage 6 (WordNet & Semantic Relationships)**, the system will evaluate whether `record` and `row` are semantically related (e.g. synsets, hypernyms).
In **Stage 7 (Triple Matching)**, the system will match `("key", "identify", "record")` with `("key", "identify", "row")` based on that semantic relation.

## 7. Summary

Stage 5 implementation is complete:
- [x] Raw dependency arcs `(governor, relation, dependent)` fully extracted.
- [x] Canonical clause relations `(subject, predicate, object)` extracted for active and passive clauses.
- [x] Surface forms and lemmas preserved.
- [x] Prepositional phrases and modifiers strictly preserved without speculative collapsing.
- [x] Zero synonym matching or grading logic introduced.